In [1]:
# کد پایتون برای اجرای رگرسیون با وقفه زمانی (EPI_lag2)
# برای روبایتنس

import pandas as pd
import statsmodels.formula.api as smf
import numpy as np

# --- 1. بارگذاری داده‌های جدید ---
# فایل اکسلی را بارگذاری می‌کنیم.
try:
    df = pd.read_excel("panel_data_with_lag.xlsx")
    print("فایل داده با موفقیت بارگذاری شد.")
    print(f"تعداد کل مشاهدات (قبل از حذف NaN): {len(df)}")
except FileNotFoundError:
    print("خطا: فایل panel_data_with_lag.xlsx یافت نشد. لطفاً مطمئن شوید فایل در همان پوشه کد قرار دارد.")
    exit()

# --- 2. آماده‌سازی داده برای رگرسیون ---
# رگرسیون نمی‌تواند با مقادیر خالی (NaN) کار کند.
# ستون EPI_lag2 برای دو سال اول هر کشور خالی است. این سطرها را حذف می‌کنیم.
df_reg = df.dropna(subset=['log_GDPpc', 'EPI_lag2', 'HDI', 'WGI_mean'])
print(f"تعداد مشاهدات معتبر برای رگرسیون (بعد از حذف NaN): {len(df_reg)}")


# --- 3. تعریف و اجرای مدل رگرسیون کوانتایل ---
# متغیر وابسته: log_GDPpc
# متغیرهای مستقل: EPI_lag2 (متغیر کلیدی با وقفه), HDI, WGI_mean
# ما مدل را برای کوانتایل‌های 10، 25، 50، 75 و 90 اجرا می‌کنیم.

quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]
results_summary = [] # لیستی برای ذخیره نتایج کلیدی

print("\n--- شروع اجرای رگرسیون کوانتایل برای هر صدک ---")

# اجرای رگرسیون در یک حلقه برای هر کوانتایل
for q in quantiles:
    print(f"\n================== نتایج برای کوانتایل {int(q*100)}% ==================")
    
    # تعریف فرمول مدل
    # این فرمول به statsmodels می‌گوید چه چیزی را بر روی چه چیزی رگرس کند
    model_formula = 'log_GDPpc ~ EPI_lag2 + HDI + WGI_mean'
    
    # ساخت و تخمین مدل رگرسیون کوانتایل
    quant_reg_model = smf.quantreg(model_formula, data=df_reg)
    result = quant_reg_model.fit(q=q)
    
    # چاپ خلاصه کامل نتایج برای بررسی
    # print(result.summary())
    
    # استخراج اطلاعات کلیدی برای جدول نهایی
    coef = result.params['EPI_lag2']
    p_value = result.pvalues['EPI_lag2']
    conf_int = result.conf_int().loc['EPI_lag2']
    
    # تعیین سطح معناداری با ستاره
    if p_value < 0.01:
        significance = '***'
    elif p_value < 0.05:
        significance = '**'
    elif p_value < 0.1:
        significance = '*'
    else:
        significance = ''
        
    results_summary.append({
        'Quantile': f"Q{int(q*100)}",
        'Coefficient': f"{coef:.4f}{significance}",
        'Std. Error': f"{result.bse['EPI_lag2']:.4f}",
        'P>|t|': f"{p_value:.4f}",
        'Conf. Int. [0.025': f"{conf_int[0]:.4f}",
        '0.975]': f"{conf_int[1]:.4f}"
    })

# --- 4. نمایش جدول خلاصه نتایج برای EPI_lag2 ---
# این جدول برای کپی کردن در مقاله شما بسیار مناسب است.
results_df = pd.DataFrame(results_summary)
print("\n\n============================================================")
print("     جدول خلاصه نتایج برای ضریب EPI_lag2")
print("============================================================")
print(results_df.to_string(index=False))
print("\nتوجه: *** p<0.01, ** p<0.05, * p<0.1")


فایل داده با موفقیت بارگذاری شد.
تعداد کل مشاهدات (قبل از حذف NaN): 1490
تعداد مشاهدات معتبر برای رگرسیون (بعد از حذف NaN): 1192

--- شروع اجرای رگرسیون کوانتایل برای هر صدک ---

================== نتایج برای کوانتایل 10% ==================

================== نتایج برای کوانتایل 25% ==================

================== نتایج برای کوانتایل 50% ==================

================== نتایج برای کوانتایل 75% ==================

================== نتایج برای کوانتایل 90% ==================


     جدول خلاصه نتایج برای ضریب EPI_lag2
Quantile Coefficient Std. Error  P>|t| Conf. Int. [0.025  0.975]
     Q10     -0.0393     0.0407 0.3355           -0.1192  0.0407
     Q25  -0.1826***     0.0484 0.0002           -0.2775 -0.0877
     Q50   -0.1231**     0.0535 0.0215           -0.2280 -0.0182
     Q75  -0.2974***     0.0676 0.0000           -0.4300 -0.1647
     Q90   -0.1892**     0.0860 0.0281           -0.3580 -0.0204

توجه: *** p<0.01, ** p<0.05, * p<0.1
